# Hermes Agent - Jupyter Notebook

This notebook allows you to interact with Hermes, the NousResearch AI agent, directly from Jupyter.

## 💾 Supabase Persistence

Your notebooks are saved to Supabase Storage, so they persist between sessions!

**Setup:** Run the cell below to connect to Supabase.

In [ ]:
# Install Supabase client
!pip install --break-system-packages supabase

In [ ]:
# Setup Supabase sync for notebook persistence
import os
import json
from supabase import create_client, Client

# Get credentials from environment
SUPABASE_URL = os.environ.get('SUPABASE_URL', '')
SUPABASE_KEY = os.environ.get('SUPABASE_KEY', '')  # anon key
BUCKET_NAME = os.environ.get('SUPABASE_BUCKET', 'notebooks')

NOTEBOOK_DIR = '/data/notebooks'

# Initialize Supabase client
if SUPABASE_URL and SUPABASE_KEY:
    supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
    print("✓ Connected to Supabase!")
else:
    supabase = None
    print("⚠️  Set SUPABASE_URL and SUPABASE_KEY environment variables")

os.makedirs(NOTEBOOK_DIR, exist_ok=True)

## 📤 Download Notebooks from Supabase

Run this cell to pull your saved notebooks:

In [ ]:
def pull_notebooks():
    """Download notebooks from Supabase Storage."""
    if not supabase:
        print("Supabase not configured")
        return
    
    os.chdir(NOTEBOOK_DIR)
    
    try:
        # List files in bucket
        files = supabase.storage.from_(BUCKET_NAME).list()
        
        if not files:
            print("No notebooks found in storage")
            return
        
        for file in files:
            if file.get('name'):
                filename = file['name']
                if filename.endswith('.ipynb'):
                    print(f"Downloading {filename}...")
                    data = supabase.storage.from_(BUCKET_NAME).download(filename)
                    with open(filename, 'wb') as f:
                        f.write(data)
        
        print(f"\n✓ Downloaded {len([f for f in files if f.get('name','').endswith('.ipynb')])} notebooks")
        print(f"Files: {os.listdir(NOTEBOOK_DIR)}")
        
    except Exception as e:
        print(f"Error: {e}")

pull_notebooks()

## ⬆️ Upload Notebooks to Supabase

Run this cell to save and upload your notebooks:

In [ ]:
def save_and_sync():
    """Upload notebooks to Supabase Storage."""
    if not supabase:
        print("Supabase not configured")
        return
    
    os.chdir(NOTEBOOK_DIR)
    uploaded = 0
    
    for filename in os.listdir('.'):
        if filename.endswith('.ipynb'):
            print(f"Uploading {filename}...")
            try:
                with open(filename, 'rb') as f:
                    supabase.storage.from_(BUCKET_NAME).upload(
                        filename,
                        f.read(),
                        {"contentType": "application/json"}
                    )
                uploaded += 1
            except Exception as e:
                # File might exist, try to update
                try:
                    supabase.storage.from_(BUCKET_NAME).update(
                        filename,
                        open(filename, 'rb').read(),
                        {"contentType": "application/json"}
                    )
                    uploaded += 1
                except Exception as e2:
                    print(f"  Error: {e2}")
    
    print(f"\n✓ Uploaded {uploaded} notebooks to Supabase")

# Run to save notebooks
save_and_sync()

---

## 🚀 Install Hermes Agent

**One-time setup:** Run this to install Hermes:

In [ ]:
# Install Hermes
!pip install --break-system-packages git+https://github.com/NousResearch/hermes-agent.git

---

## Initialize Hermes Agent

In [ ]:
from run_agent import AIAgent

# Initialize the agent
# Set your API key: os.environ['OPENAI_API_KEY'] = 'your-key'

agent = AIAgent(
    model="openai/gpt-4o",  # Change to your preferred model
    quiet_mode=True,
)

print("Hermes Agent initialized!")

## Chat with Hermes

In [ ]:
response = agent.chat("Hello! What can you help me with?")
print(response)